# Finetuning LVH

Running `AS_detection/main.py` in this notebook.


## Setup

In [1]:
import os
import shutil
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torchvision
import tqdm
from sklearn import metrics

LVH_DIR = Path.cwd() / "LVH_detection"
sys.path.insert(0, str(LVH_DIR))

from dataset import EchoDatasetLVHDetection
from utils import evaluate, seed_worker, set_seed, train, validate

In [2]:
# Hyperparameters (defaults match the argparse options in main.py)
args = SimpleNamespace(
    data_dir="data",
    output_dir="output",
    clip_len=16,
    sampling_rate=1,
    num_clips=4,
    model_name="3dresnet18",
    rand_init=False,
    n_gpu=1,
    max_epochs=10,
    patience=1e4,
    lr=1e-4,
    batch_size=10,
    label_smoothing=0.0,
    use_class_weights=False,
    dropout_fc=False,
    augment=False,
    lpft="",
    ssl="ssl_pretraining/output/model-2026-1002",
    frac=1.0,
    weight_averaging=False,
    n_TTA=0,
    seed=0,
)
args

namespace(data_dir='data',
          output_dir='output',
          clip_len=16,
          sampling_rate=1,
          num_clips=4,
          model_name='3dresnet18',
          rand_init=False,
          n_gpu=1,
          max_epochs=10,
          patience=10000.0,
          lr=0.0001,
          batch_size=10,
          label_smoothing=0.0,
          use_class_weights=False,
          dropout_fc=False,
          augment=False,
          lpft='',
          ssl='ssl_pretraining/output/model-2026-1002',
          frac=1.0,
          weight_averaging=False,
          n_TTA=0,
          seed=0)

In [3]:
MODEL_NAME = args.model_name
MODEL_NAME += f"_frac-{args.frac}" if args.frac != 1.0 else ""
MODEL_NAME += "_pretr" if not args.rand_init else "_rand"
MODEL_NAME += f"_ssl-{args.ssl.split('/')[-1]}" if args.ssl != "" else ""
MODEL_NAME += "_aug" if args.augment else ""
MODEL_NAME += f"_clip-len-{args.clip_len}-stride-{args.sampling_rate}"
MODEL_NAME += f"_num-clips-{args.num_clips}"
MODEL_NAME += "_cw" if args.use_class_weights else ""
MODEL_NAME += f"_lr-{args.lr}"
MODEL_NAME += f"_{args.max_epochs}ep"
MODEL_NAME += f"_patience-{args.patience}" if args.patience != 1e4 else ""
MODEL_NAME += f"_bs-{args.batch_size}"
MODEL_NAME += f"_ls-{args.label_smoothing}" if args.label_smoothing != 0.0 else ""
MODEL_NAME += f"_drp-0.25" if args.dropout_fc else ""
MODEL_NAME += f"_weight-avg" if args.weight_averaging else ""
MODEL_NAME += f"_seed-{args.seed}" if args.seed != 0 else ""

MODEL_NAME += "_val-auc"

MODEL_NAME

'3dresnet18_pretr_ssl-model-2026-1002_clip-len-16-stride-1_num-clips-4_lr-0.0001_10ep_bs-10_val-auc'

In [4]:
# Create output directory for model (and delete if already exists)
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

model_dir = os.path.join(args.output_dir, MODEL_NAME)
if os.path.isdir(model_dir):
    shutil.rmtree(model_dir)
os.mkdir(model_dir)

model_dir

'output/3dresnet18_pretr_ssl-model-2026-1002_clip-len-16-stride-1_num-clips-4_lr-0.0001_10ep_bs-10_val-auc'

In [5]:
# Set all seeds for reproducibility
set_seed(args.seed)

# Set device (main.py hardcodes cuda:0; fall back so this also runs on a Mac)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='mps')

## Create Model

In [6]:
# Create model
if args.model_name == "3dresnet18":
    weights = (
        torchvision.models.video.R3D_18_Weights.DEFAULT if not args.rand_init else None
    )
    model = torchvision.models.video.r3d_18(weights=weights)

    if args.dropout_fc:
        model.fc = torch.nn.Sequential(  # type: ignore
            torch.nn.Linear(512, 1), torch.nn.Dropout(0.25)
        )

        if args.lpft != "":
            # Linear probing fine-tuning (LPFT) weights are stored in a dict under the key "weights"
            weights = dict(torch.load(args.lpft, map_location="cpu")["weights"])

            model.fc[0].weight.data = weights["fc.0.weight"]
            model.fc[0].bias.data = weights["fc.0.bias"]

    else:
        model.fc = torch.nn.Linear(512, 1)
        if args.lpft != "":
            # Linear probing fine-tuning (LPFT) weights are stored in a dict under the key "weights"
            weights = dict(torch.load(args.lpft, map_location="cpu")["weights"])

            model.fc.weight.data = weights["fc.weight"]
            model.fc.bias.data = weights["fc.bias"]

    if args.ssl != "":
        checkpoints = [f for f in os.listdir(args.ssl) if f.endswith(".pt")]
        idx = np.argmax([int(f.split(".")[0].split("-")[1]) for f in checkpoints])
        weights = torch.load(
            os.path.join(args.ssl, checkpoints[idx]), map_location="cpu"
        )["weights"]
        weights = {
            k.replace("encoder.", ""): v for k, v in weights.items() if "encoder" in k
        }

        model.load_state_dict(weights, strict=False)


model = model.to(device)

In [7]:
model

VideoResNet(
  (stem): BasicStem(
    (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (1): BasicBlock(
      (conv1): Sequential(
        (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1

## Create Dataset

In [8]:
train_dataset = EchoDatasetLVHDetection(
    data_dir=args.data_dir,
    split="train",
    clip_len=args.clip_len,
    sampling_rate=args.sampling_rate,
    num_clips=args.num_clips,
    augment=args.augment,
    kinetics=(args.ssl == "") and (not args.rand_init),
)

LVH class distribution (train split):
lvh
0    0.8
1    0.2
Name: proportion, dtype: float64


In [9]:
train_dataset[0]["x"]

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         ...,

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 

In [10]:
print(train_dataset.label_df.shape)
train_dataset.label_df.columns.tolist()

(10, 7)


['HashedFileName', 'split', 'IVSd', 'LVIDd', 'LVPWd', 'max_wall', 'lvh']

In [11]:
print(train_dataset.label_df.head())
print(train_dataset.video_dir)
print(os.listdir(train_dataset.video_dir)[:5])

       HashedFileName  split      IVSd     LVIDd     LVPWd  max_wall  lvh
0  0X1A3E19F2A68FA5C9  train  1.066211  4.886401  0.737920  1.066211    0
1  0X1AC9C81F31EACA8C  train  1.455638  4.622425  0.755547  1.455638    1
2  0X1B6E9380426F9B6C  train  0.919435  4.091271  0.883404  0.919435    0
3  0X1A1089806A165C77  train  1.358560  4.438374  1.348886  1.358560    1
4  0X1A0EA15E24B40C14  train  0.979501  5.652974  0.820009  0.979501    0
data/videos
['0X1A5B78D3E3ED2356.avi', '0X1A026FE97C81DF79.avi', '0X1B9CCA10167CE4AE.avi', '0X1A2FBB94F59A4246.avi', '0X1A3E19F2A68FA5C9.avi']


## Create DataLoader

In [12]:
# Create loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=args.n_gpu * args.batch_size,
    shuffle=True,
    num_workers=8,
    worker_init_fn=seed_worker,
)

## Initialize Model Training Loop

In [13]:
# Create csv documenting training history
history = pd.DataFrame(
    columns=[
        "epoch",
        "phase",
        "loss",
        "auroc",
        "aupr",
        "acc",
        "b_acc",
        "mcc",
        "precision",
        "recall",
        "f1",
    ]
)
history.to_csv(os.path.join(model_dir, "history.csv"), index=False)

if args.use_class_weights:
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.sort(
            np.unique((train_dataset.label_df["av_stenosis"] == "Severe").values)
        ),
        y=(train_dataset.label_df["av_stenosis"] == "Severe").values,
    )

    print("Class weights:", class_weights)

    pos_weight = (class_weights / class_weights.min())[1]

    print("Normalized positive class weight:", pos_weight)

    loss_fxn = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.Tensor([pos_weight]).to(device)
    )
    eval_loss_fxn = torch.nn.BCELoss(weight=torch.Tensor([pos_weight]).to(device))
else:
    loss_fxn = torch.nn.BCEWithLogitsLoss()
    eval_loss_fxn = torch.nn.BCELoss()

# Create optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

## Train Model

In [14]:
data_loader = train_loader
epoch = 1

In [15]:
model.train()

pbar = tqdm.tqdm(enumerate(data_loader), total=len(data_loader), desc=f"Epoch {epoch}")
running_loss = 0.0
y_true, y_hat = [], []
acc_nums = []
plax_probs = []

Epoch 1:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
train_dataset[0]["x"].shape

torch.Size([3, 16, 112, 112])

In [17]:
for i, batch in pbar:
        x, y = batch['x'].to(device), batch['y'].to(device)
        print(batch)
        acc_num = batch['acc_num']
        print(acc_num)

Epoch 1: 100%|██████████| 1/1 [00:13<00:00, 13.53s/it]

{'x': tensor([[[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],

          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],

          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],

          ...,

          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
     

Epoch 1: 100%|██████████| 1/1 [00:18<00:00, 18.54s/it]


In [ ]:
x, y = batch['x'].to(device), batch['y'].to(device)
acc_num = batch['acc_num']

# Forward pass
out = model(x)

# Compute loss
loss = loss_fxn(out, y)

# Backward pass
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()

In [ ]:
# Train with early stopping
epoch = 1
early_stopping_dict = {"best_auroc": 0.0, "epochs_no_improve": 0}
best_model_wts = None
while (
    epoch <= args.max_epochs
    and early_stopping_dict["epochs_no_improve"] < args.patience
):
    history = train(
        model=model,
        device=device,
        loss_fxn=loss_fxn,
        optimizer=optimizer,
        data_loader=train_loader,
        history=history,
        epoch=epoch,
        model_dir=model_dir,
    )
    history, early_stopping_dict, best_model_wts = validate(
        model=model,
        device=device,
        loss_fxn=eval_loss_fxn,
        optimizer=optimizer,
        data_loader=val_loader,
        history=history,
        epoch=epoch,
        model_dir=model_dir,
        early_stopping_dict=early_stopping_dict,
        best_model_wts=best_model_wts,
    )

    epoch += 1

if args.weight_averaging:
    checkpoints = [
        torch.load(os.path.join(model_dir, f), map_location="cpu")["weights"]
        for f in os.listdir(model_dir)
        if f.endswith(".pt")
    ]
    for key in best_model_wts:
        best_model_wts[key] = torch.stack(
            [chkpt[key] for chkpt in checkpoints], dim=0
        ).sum(dim=0) / len(checkpoints)


## Validate Model